In [1]:

import pandas as pd

df = pd.read_excel("Superstore dataset (1).xlsx")

print("Dataset loaded successfully!")


Dataset loaded successfully!


In [2]:

df.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:

import os

print(os.listdir())


['.config', 'Superstore dataset (1).xlsx', 'sample_data']


In [4]:

# Check the number of rows and columns
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

# Check data types and non-null counts
print("\nDataset Information:")
df.info()


Dataset Shape: (9994, 21)

Column Names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   object        
 2   Order Date     9994 non-null   datetime64[ns]
 3   Ship Date      9994 non-null   datetime64[ns]
 4   Ship Mode      9994 non-null   object        
 5   Customer ID    9994 non-null   object        
 6   Customer Name  9994 non-null   object        
 7   Segment        9994 non-null   object        
 8   Country        9994 non-null   object        
 9   City

In [5]:

# Check missing values in each column
missing_values = df.isnull().sum()

print("Missing Values in Each Column:")
print(missing_values)

# Calculate total missing values
print("\nTotal Missing Values:", missing_values.sum())


Missing Values in Each Column:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

Total Missing Values: 0


In [6]:

duplicate_count = df.duplicated().sum()

print("Total Duplicate Rows:", duplicate_count)


Total Duplicate Rows: 0


In [8]:

print("Current Data Types:")
print(df.dtypes)

print("\nDate Column Checks:")
print("Order Date:", pd.api.types.is_datetime64_any_dtype(df["Order Date"]))
print("Ship Date:", pd.api.types.is_datetime64_any_dtype(df["Ship Date"]))


Current Data Types:
Row ID                    int64
Order ID                 object
Order Date       datetime64[ns]
Ship Date        datetime64[ns]
Ship Mode                object
Customer ID              object
Customer Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal Code               int64
Region                   object
Product ID               object
Category                 object
Sub-Category             object
Product Name             object
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object

Date Column Checks:
Order Date: True
Ship Date: True


In [9]:

# Check values that may be incorrect
print("Invalid sales:", (df["Sales"] <= 0).sum())
print("Invalid quantity:", (df["Quantity"] <= 0).sum())
print("Invalid discount:", ((df["Discount"] < 0) | (df["Discount"] > 1)).sum())
print("Ship date before order date:", (df["Ship Date"] < df["Order Date"]).sum())
print("Negative profit:", (df["Profit"] < 0).sum())


Invalid sales: 0
Invalid quantity: 0
Invalid discount: 0
Ship date before order date: 0
Negative profit: 1871


In [10]:

# Check repeated order-product combinations
print(
    df.duplicated(subset=["Order ID", "Product ID"]).sum()
)


8


In [11]:

# Find blank text cells
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    blanks = df[col].apply(
        lambda x: isinstance(x, str) and x.strip() == ""
    ).sum()
    if blanks > 0:
        print(col, ":", blanks)


In [18]:

# Check data quality rules
rules = {
    "Missing values": df.isnull().any(axis=1),
    "Duplicate rows": df.duplicated(),
    "Invalid sales": df["Sales"] <= 0,
    "Invalid quantity": df["Quantity"] <= 0,
    "Invalid discount": (df["Discount"] < 0) | (df["Discount"] > 1),
    "Ship date before order date": df["Ship Date"] < df["Order Date"],
    "Negative profit": df["Profit"] < 0
}

# Count records for each rule
results = []

for issue, condition in rules.items():
    results.append({
        "Issue": issue,
        "Affected Rows": int(condition.sum()),
        "Rule": {
            "Missing values": "No required values should be missing",
            "Duplicate rows": "Rows should not be exact duplicates",
            "Invalid sales": "Sales must be greater than zero",
            "Invalid quantity": "Quantity must be greater than zero",
            "Invalid discount": "Discount must be between 0 and 1",
            "Ship date before order date": "Ship date must be on or after order date",
            "Negative profit": "Review losses; negative profit is allowed"
        }[issue]
    })

validation_report = pd.DataFrame(results)
display(validation_report)



,Issue,Affected Rows,Rule
0,Missing values,0,No required values should be missing
1,Duplicate rows,0,Rows should not be exact duplicates
2,Invalid sales,0,Sales must be greater than zero
3,Invalid quantity,0,Quantity must be greater than zero
4,Invalid discount,0,Discount must be between 0 and 1
5,Ship date before order date,0,Ship date must be on or after order date
6,Negative profit,1871,Review losses; negative profit is allowed


In [19]:

# Prepare the issue log
issue_log = validation_report.copy()

# Add status for each issue
issue_log["Status"] = issue_log.apply(
    lambda row: (
        "Review"
        if row["Affected Rows"] > 0
        else "No issues found"
    ),
    axis=1
)

# Negative profit can be a normal business loss
issue_log.loc[
    issue_log["Issue"] == "Negative profit",
    "Status"
] = "Review business losses"

display(issue_log)


,Issue,Affected Rows,Rule,Status
0,Missing values,0,No required values should be missing,No issues found
1,Duplicate rows,0,Rows should not be exact duplicates,No issues found
2,Invalid sales,0,Sales must be greater than zero,No issues found
3,Invalid quantity,0,Quantity must be greater than zero,No issues found
4,Invalid discount,0,Discount must be between 0 and 1,No issues found
5,Ship date before order date,0,Ship date must be on or after order date,No issues found
6,Negative profit,1871,Review losses; negative profit is allowed,Review business losses


In [21]:

clean_df = df.copy()

clean_df = clean_df.drop_duplicates()

# Remove extra spaces from text values
text_cols = clean_df.select_dtypes(include="object").columns

for col in text_cols:
    clean_df[col] = clean_df[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Original rows:", len(df))
print("Cleaned rows:", len(clean_df))
print("Remaining duplicates:", clean_df.duplicated().sum())



Original rows: 9994
Cleaned rows: 9994
Remaining duplicates: 0


In [20]:

# Summarize the audit results
audit_report = pd.DataFrame({
    "Metric": [
        "Total rows",
        "Total columns",
        "Missing values",
        "Duplicate rows",
        "Invalid sales",
        "Invalid quantity",
        "Invalid discount",
        "Ship date before order date",
        "Negative profit",
        "Cleaned rows"
    ],
    "Value": [
        len(df),
        len(df.columns),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        int((df["Sales"] <= 0).sum()),
        int((df["Quantity"] <= 0).sum()),
        int(((df["Discount"] < 0) | (df["Discount"] > 1)).sum()),
        int((df["Ship Date"] < df["Order Date"]).sum()),
        int((df["Profit"] < 0).sum()),
        len(df.drop_duplicates())
    ]
})

display(audit_report)



,Metric,Value
0,Total rows,9994
1,Total columns,21
2,Missing values,0
3,Duplicate rows,0
4,Invalid sales,0
5,Invalid quantity,0
6,Invalid discount,0
7,Ship date before order date,0
8,Negative profit,1871
9,Cleaned rows,9994


In [22]:

with pd.ExcelWriter("Superstore_Audit_Report.xlsx") as writer:
    audit_report.to_excel(
        writer, sheet_name="Audit Report", index=False
    )
    issue_log.to_excel(
        writer, sheet_name="Issue Log", index=False
    )

print("Audit report saved!")


Audit report saved!


In [23]:

clean_df.to_csv("Superstore_Cleaned.csv", index=False)

print("Cleaned dataset saved!")


Cleaned dataset saved!


In [24]:

from google.colab import files

files.download("Superstore_Audit_Report.xlsx")
files.download("Superstore_Cleaned.csv")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>